# Assignement - 14 : Feature Engineering, Encoding, Scaling & Pipelines

## Name - Suryansh Tiwari

## Part - 1

In [1]:
import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [2]:
df=pd.read_csv('retail_sales_dataset.csv')
df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"])
print(df.head())

   Transaction ID       Date  ... Price per Unit Total Amount
0               1 2023-11-24  ...             50          150
1               2 2023-02-27  ...            500         1000
2               3 2023-01-13  ...             30           30
3               4 2023-05-21  ...            500          500
4               5 2023-05-06  ...             50          100

[5 rows x 9 columns]


In [3]:
print("Shape of Dataset:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Shape of Dataset:
(1000, 9)

Column Names:
Index(['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age',
       'Product Category', 'Quantity', 'Price per Unit', 'Total Amount'],
      dtype='str')

Data Types:
Transaction ID               int64
Date                datetime64[us]
Customer ID                    str
Gender                         str
Age                          int64
Product Category               str
Quantity                     int64
Price per Unit               int64
Total Amount                 int64
dtype: object

Missing Values:
Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64


## Task-1 : Creating new features

In [5]:
df['Total Value'] = df['Quantity'] * df['Price per Unit']

df['Age Group'] = pd.cut(
    df['Age'],
    bins=[0,25,40,60,100],
    labels=["Young","Adult","Middle Age", "Senior"]
)
df["Revenue Category"] = pd.cut(
    df["Total Amount"],
    bins=[0,500,1000,1500,3000],
    labels=["low","medium","high","Very high"]
)
df['Discount Percentage'] = (
    (df['Total Value']- df['Total Amount'])/ df['Total Value']
) * 100
print(df.head())

   Transaction ID       Date  ... Revenue Category Discount Percentage
0               1 2023-11-24  ...              low                 0.0
1               2 2023-02-27  ...           medium                 0.0
2               3 2023-01-13  ...              low                 0.0
3               4 2023-05-21  ...              low                 0.0
4               5 2023-05-06  ...              low                 0.0

[5 rows x 13 columns]


## Task-2 : Date Feature Engineering

In [6]:
df["Year"]= df["Date"].dt.year
df["Month"]= df["Date"].dt.month
df["Day"]= df["Date"].dt.day

print(df[["Date","Year","Month","Day"]].head())

        Date  Year  Month  Day
0 2023-11-24  2023     11   24
1 2023-02-27  2023      2   27
2 2023-01-13  2023      1   13
3 2023-05-21  2023      5   21
4 2023-05-06  2023      5    6


## Task-3 : One Hot Encoding using pd.get_dummies()

In [7]:
categorical_columns = [
    "Gender", "Product Category", "Age Group", "Revenue Category"
]
encoded_df = pd.get_dummies(df, columns=categorical_columns)
print(encoded_df.head())

   Transaction ID       Date  ... Revenue Category_high  Revenue Category_Very high
0               1 2023-11-24  ...                 False                       False
1               2 2023-02-27  ...                 False                       False
2               3 2023-01-13  ...                 False                       False
3               4 2023-05-21  ...                 False                       False
4               5 2023-05-06  ...                 False                       False

[5 rows x 25 columns]


## Task-4 : Column Transfer

In [8]:
numerical_features = [
    "Age", "Quantity", "Price per Unit", "Total Value"
]
categorical_features = [
    "Gender", "Product Category","Age Group"
]

In [10]:
preprocessor = ColumnTransformer(transformers=[
    (
        "cat",
        OneHotEncoder(),
        categorical_features
    ),
    (
        "num",
        "passthrough",
        numerical_features
    )
])
transformed_data = preprocessor.fit_transform(df)
print(transformed_data)

[[   0.    1.    1. ...    3.   50.  150.]
 [   1.    0.    0. ...    2.  500. 1000.]
 [   0.    1.    0. ...    1.   30.   30.]
 ...
 [   1.    0.    1. ...    4.   25.  100.]
 [   1.    0.    0. ...    3.   50.  150.]
 [   0.    1.    0. ...    4.   30.  120.]]


## Task-5 : StandardScaler

In [11]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[numerical_features])
scaled_df = pd.DataFrame(scaled_data, columns=numerical_features)
print(scaled_df.head())

print("\nMeans:")
print(scaled_df.mean())

print("\nStandard Deviations:")
print(scaled_df.std())

        Age  Quantity  Price per Unit  Total Value
0 -0.540565  0.429265       -0.685123    -0.546704
1 -1.125592 -0.453996        1.688464     0.971919
2  0.629489 -1.337258       -0.790615    -0.761098
3 -0.321180 -1.337258        1.688464     0.078611
4 -0.833078 -0.453996       -0.685123    -0.636035

Means:
Age              -2.344791e-16
Quantity          1.794120e-16
Price per Unit    5.329071e-17
Total Value      -3.552714e-18
dtype: float64

Standard Deviations:
Age               1.0005
Quantity          1.0005
Price per Unit    1.0005
Total Value       1.0005
dtype: float64


## Task-6 : MinMaxScaler

In [12]:
minmax = MinMaxScaler()

minmax_data = minmax.fit_transform(df[numerical_features])

minmax_df = pd.DataFrame(minmax_data, columns=numerical_features)
print(minmax_df.head())

        Age  Quantity  Price per Unit  Total Value
0  0.347826  0.666667        0.052632     0.063291
1  0.173913  0.333333        1.000000     0.493671
2  0.695652  0.000000        0.010526     0.002532
3  0.413043  0.000000        1.000000     0.240506
4  0.260870  0.333333        0.052632     0.037975


In [13]:
print("StandardScaler Output")
print(scaled_df.head())

print("MinMaxScaler Output")
print(minmax_df.head())

StandardScaler Output
        Age  Quantity  Price per Unit  Total Value
0 -0.540565  0.429265       -0.685123    -0.546704
1 -1.125592 -0.453996        1.688464     0.971919
2  0.629489 -1.337258       -0.790615    -0.761098
3 -0.321180 -1.337258        1.688464     0.078611
4 -0.833078 -0.453996       -0.685123    -0.636035
MinMaxScaler Output
        Age  Quantity  Price per Unit  Total Value
0  0.347826  0.666667        0.052632     0.063291
1  0.173913  0.333333        1.000000     0.493671
2  0.695652  0.000000        0.010526     0.002532
3  0.413043  0.000000        1.000000     0.240506
4  0.260870  0.333333        0.052632     0.037975


## Task-7 Create Preprocessing Pipeline

In [14]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [15]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder()
        )
    ]
)

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)
print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['Age', 'Quantity', 'Price per Unit',
                                  'Total Value']),
                                ('cat',
                                 Pipeline(steps=[('encoder', OneHotEncoder())]),
                                 ['Gender', 'Product Category', 'Age Group'])])


## Task-8 : Full Scikit-learn Pipeline

In [17]:
X = df[
    numerical_features +
    categorical_features
]

y = df["Total Amount"]

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42

)

In [18]:
model_pipeline = Pipeline(

    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            LinearRegression()
        )

    ]

)

In [19]:
model_pipeline.fit(
    X_train,
    y_train
)

print("Model Trained Successfully")

Model Trained Successfully


In [20]:
predictions = model_pipeline.predict(
    X_test
)

print("Predictions")

print(predictions[:10])

Predictions
[1500.  100.  300.  100. 2000.   90.   50.  300.  200. 1000.]


## Task-9 : Conceptual Questions

### 1. Why are pipelines important in Machine Learning?

Pipelines automate preprocessing and model training in a single workflow. They make the code cleaner, reusable, and reduce the chances of errors.

### 2. What problems do pipelines solve?

Pipelines ensure that the same preprocessing steps are applied consistently to both training and testing data. They also help prevent data leakage and simplify model deployment.

### 3. Difference between manual preprocessing and pipeline-based preprocessing.

Manual preprocessing requires each step to be performed separately, making the process more error-prone and harder to maintain. Pipeline-based preprocessing combines all steps into one reusable workflow, improving consistency, readability, and scalability.